<a href="https://colab.research.google.com/github/rcNibedita/De-Novo-Gen/blob/main/03_DeNovo_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 3 — MolGPT Molecular Generation
## Simple, robust baseline + two scaffold experiments

### Goal

**Baseline A:** unconditional MolGPT generation.

**Experiment 1:** 20 diversity-selected BM scaffolds → MolGPT SMILES-prefix continuation.

**Experiment 2:** the same 20 BM scaffolds are treated as fixed cores. RDKit determines the **observed attachment positions** and labels them (`R1`, `R2`, ...). Valid substituents are collected separately for each position and recombined only at the matching labelled positions.

This version does **not** ask MolGPT to generate incomplete fragments, and it does **not** attach substituents to an arbitrary hydrogen-bearing atom.

TAK1 protein structure/binding-pocket information is intentionally not used here; that belongs to the later structure-guided notebook.


## 1. Install packages

Designed for Google Colab.

In [ ]:
!pip -q install rdkit transformers huggingface_hub scikit-learn pandas matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 13.5 MB/s eta 0:00:00


In [ ]:
# Mount Google Drive

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## 2. Imports and reproducibility

In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, Crippen, Lipinski, QED, rdMolDescriptors, AllChem
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

import torch
from transformers import GPT2LMHeadModel, PreTrainedTokenizerFast
from huggingface_hub import hf_hub_download

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cpu


## 3. Load the cleaned TAK1 inhibitor dataset

Expected file:

`/content/TAK1_clean_canonical.csv`

The notebook also uses an existing `df_canonical` dataframe if it is already present.

In [ ]:
DATA_PATH = "/content/drive/MyDrive/De-Novo/TAK1_clean_canonical.csv"

if "df_canonical" in globals():
    tak1_df = df_canonical.copy()
elif os.path.exists(DATA_PATH):
    tak1_df = pd.read_csv(DATA_PATH)
else:
    raise FileNotFoundError(
        "TAK1_clean_canonical.csv was not found. "
        "Upload the Notebook 1 output to Colab or define df_canonical."
    )

if "Canonical_SMILES" in tak1_df.columns:
    SMILES_COL = "Canonical_SMILES"
elif "Smiles" in tak1_df.columns:
    SMILES_COL = "Smiles"
else:
    raise KeyError("Expected Canonical_SMILES or Smiles column.")

def to_mol(s):
    if not isinstance(s, str) or not s.strip():
        return None
    try:
        return Chem.MolFromSmiles(s.strip())
    except Exception:
        return None

tak1_df = tak1_df.dropna(subset=[SMILES_COL]).copy()
tak1_df["Mol"] = tak1_df[SMILES_COL].apply(to_mol)
tak1_df = tak1_df[tak1_df["Mol"].notna()].copy()
tak1_df["SMILES"] = tak1_df["Mol"].apply(Chem.MolToSmiles)
tak1_df = tak1_df.drop_duplicates("SMILES").reset_index(drop=True)

print("Valid unique TAK1 reference molecules:", len(tak1_df))

Valid unique TAK1 reference molecules: 178


## 4. Extract BM scaffolds

Peripheral substituents are removed to obtain the BM scaffold.

In [ ]:
def get_bm_smiles(mol):
    try:
        scaf = MurckoScaffold.GetScaffoldForMol(mol)
        if scaf is None or scaf.GetNumAtoms() == 0:
            return None
        return Chem.MolToSmiles(scaf)
    except Exception:
        return None

tak1_df["BM_Scaffold"] = tak1_df["Mol"].apply(get_bm_smiles)

scaffold_df = (
    tak1_df.dropna(subset=["BM_Scaffold"])
    .drop_duplicates("BM_Scaffold")
    .reset_index(drop=True)
)

print("Unique BM scaffolds:", len(scaffold_df))

Unique BM scaffolds: 104


## 5. Select 20 diverse BM scaffolds

We do **not** use `head(20)`.

Morgan fingerprints and a deterministic MaxMin-style procedure are used to spread the selected seeds through scaffold chemical space.

In [ ]:
MAX_SEEDS = 20
fp_gen = AllChem.GetMorganGenerator(radius=2, fpSize=2048)

scaffold_mols = []
scaffold_smiles = []

for smi in scaffold_df["BM_Scaffold"]:
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        scaffold_smiles.append(smi)
        scaffold_mols.append(mol)

fps = [fp_gen.GetFingerprint(m) for m in scaffold_mols]
n = len(fps)

if n <= MAX_SEEDS:
    selected = list(range(n))
else:
    sim = np.eye(n)
    for i in range(n):
        for j in range(i + 1, n):
            x = DataStructs.TanimotoSimilarity(fps[i], fps[j])
            sim[i, j] = sim[j, i] = x

    mean_sim = (sim.sum(axis=1) - 1) / (n - 1)
    selected = [int(np.argmin(mean_sim))]
    remaining = set(range(n)) - set(selected)

    while len(selected) < MAX_SEEDS:
        best = max(
            remaining,
            key=lambda i: 1 - max(sim[i, j] for j in selected)
        )
        selected.append(best)
        remaining.remove(best)

selected_scaffolds = pd.DataFrame({
    "Seed_ID": [f"Seed_{i+1:02d}" for i in range(len(selected))],
    "Scaffold": [scaffold_smiles[i] for i in selected]
})

print("Selected seeds:", len(selected_scaffolds))
display(selected_scaffolds)

Selected seeds: 20


,Seed_ID,Scaffold
0,Seed_01,O=C1/C=C\CCOC(=O)C2=C(/C=C\CCC1)CC=CC2=O
1,Seed_02,c1ncc2cc[nH]c2n1
2,Seed_03,c1csc(-c2ccc(CN3CCNCC3)cc2)c1
3,Seed_04,O=C(N=c1[nH]c2ccccc2[nH]1)c1ccccc1
4,Seed_05,c1cncc(-c2c(-c3cccc4nnsc34)oc3cncc(-c4cnn(C5CC...
5,Seed_06,O=C(CCCC[C@@H]1SC[C@@H]2NC(=O)N[C@@H]21)NCCOCC...
6,Seed_07,O=C1Nc2ccccc2OC[C@@H]1N1CCc2cn(CC3CCS(=O)(=O)C...
7,Seed_08,c1cc2[nH]ncc2cc1-c1cnc2ccc(N3C[C@@H]4C[C@H]3CO...
8,Seed_09,O=C1NCc2c1c1c3ccccc3n3c1c1c2c2ccccc2n1C1CCC[C@...
9,Seed_10,O=C(Nc1cccc(-c2nnc[nH]2)n1)Nc1nccc2ccccc12


## 6. Load MolGPT

Checkpoint used in the previous notebook:

`jonghyunlee/MolGPT_pretrained-by-ZINC15`

We only use inference; no fine-tuning occurs here.

In [ ]:
MODEL_NAME = "jonghyunlee/MolGPT_pretrained-by-ZINC15"

tokenizer_path = hf_hub_download(
    repo_id=MODEL_NAME,
    filename="tokenizer.json"
)

tokenizer = PreTrainedTokenizerFast(tokenizer_file=tokenizer_path)
tokenizer.model_max_length = 128
tokenizer.pad_token = "<pad>"
tokenizer.bos_token = "<bos>"
tokenizer.eos_token = "<eos>"

model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

MODEL_MAX_LEN = int(getattr(model.config, "n_positions", 128))

print("Model max positions:", MODEL_MAX_LEN)
print("Tokenizer vocab:", tokenizer.vocab_size)

tokenizer.json:   0%|          | 0.00/149k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/822 [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 2139), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 2139), got 50256. This may result in unexpected behavior.


model.safetensors: reconstructing file:   0%|          |  0.00B / 27.6MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

Model max positions: 128
Tokenizer vocab: 2140


## 7. Generation settings

These numbers are deliberately small so the notebook remains practical and easy to inspect.

In [ ]:
N_UNCOND = 100
N_PER_SEED = 10

TEMPERATURE = 1.0
TOP_K = 50
TOP_P = 0.95

MAX_NEW_TOKENS_UNCOND = 64
MAX_NEW_TOKENS_SEEDED = 64

print("Unconditional:", N_UNCOND)
print("Seeded:", len(selected_scaffolds) if False else len(selected_scaffolds), "scaffolds ×", N_PER_SEED)

Unconditional: 100
Seeded: 20 scaffolds × 10


# Baseline A — Unconditional generation

No TAK1-derived scaffold is given to MolGPT.

In [ ]:
@torch.no_grad()
def generate_unconditional(n, max_new_tokens=64):
    input_ids = torch.tensor(
        [[tokenizer.bos_token_id]],
        dtype=torch.long,
        device=DEVICE
    )

    outputs = model.generate(
        input_ids=input_ids,
        max_new_tokens=max_new_tokens,
        num_return_sequences=n,
        do_sample=True,
        temperature=TEMPERATURE,
        top_k=TOP_K,
        top_p=TOP_P,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    return [
        tokenizer.decode(x, skip_special_tokens=True).strip()
        for x in outputs
    ]

raw_uncond = generate_unconditional(N_UNCOND, MAX_NEW_TOKENS_UNCOND)

def clean_smiles_list(smiles_list):
    rows = []
    for s in smiles_list:
        mol = to_mol(s)
        if mol is not None:
            rows.append({
                "SMILES": Chem.MolToSmiles(mol),
                "Mol": mol
            })
    if not rows:
        return pd.DataFrame(columns=["SMILES", "Mol"])
    return pd.DataFrame(rows).drop_duplicates("SMILES").reset_index(drop=True)

df_uncond = clean_smiles_list(raw_uncond)

print("Raw:", len(raw_uncond))
print("Valid unique:", len(df_uncond))
display(df_uncond.head(10))

Raw: 100
Valid unique: 100


,SMILES,Mol
0,C[C@H]1C[C@@H](NC(=O)c2cnn(C)n2)CN1C(=O)c1ccc(...,<rdkit.Chem.rdchem.Mol object at 0x7c20d1e8b680>
1,CCC[C@@H](O)[C@@H](CO)NC(=O)C(=O)Nc1cccc(C#N)n1,<rdkit.Chem.rdchem.Mol object at 0x7c20d1e8aea0>
2,C[C@H](C(=O)N(C)C1CN(C(=O)Cc2ccnn2C)C1)N1C(=O)...,<rdkit.Chem.rdchem.Mol object at 0x7c20d1e8af10>
3,Cc1ncc(C(=O)N[C@H]2C[C@@H](NC(=O)[C@H]3COCCO3)...,<rdkit.Chem.rdchem.Mol object at 0x7c20d0211a80>
4,O=C1NCC[C@H]1N1CC[C@@H](N2CCN(C(=O)CCc3cnn[nH]...,<rdkit.Chem.rdchem.Mol object at 0x7c20d0211a10>
5,CNC(=O)NC(=O)CNC[C@@H]1CCCCN1C(=O)c1ccncn1,<rdkit.Chem.rdchem.Mol object at 0x7c20d02119a0>
6,CC[C@@H](CNS(C)(=O)=O)NC(=O)N[C@H]1COC[C@@H]1OC,<rdkit.Chem.rdchem.Mol object at 0x7c20d0211930>
7,C[C@H](CNC(=O)c1ccc(-c2nn[nH]n2)[nH]1)N(C)CC(=...,<rdkit.Chem.rdchem.Mol object at 0x7c20d02118c0>
8,CN(CC(=O)N[C@H]1CCS(=O)(=O)C1)C(=O)CNC(=O)c1cc...,<rdkit.Chem.rdchem.Mol object at 0x7c20d02117e0>
9,C[C@@H]1CC[C@@H]1N[C@@H]1CCN(C(N)=O)C1,<rdkit.Chem.rdchem.Mol object at 0x7c20d0211770>


# Experiment 1 — BM scaffold-prefix generation

Each BM scaffold is supplied as a SMILES prefix.

**Important:** this is ordinary sequence continuation. It is not explicit substituent placement.

In [ ]:
@torch.no_grad()
def generate_seeded(seed_smiles, n, max_new_tokens=64):
    prompt = tokenizer.bos_token + seed_smiles
    encoded = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    )
    input_ids = encoded["input_ids"].to(DEVICE)

    available = MODEL_MAX_LEN - input_ids.shape[1]
    if available < 1:
        return []

    outputs = model.generate(
        input_ids=input_ids,
        max_new_tokens=min(max_new_tokens, available),
        num_return_sequences=n,
        do_sample=True,
        temperature=TEMPERATURE,
        top_k=TOP_K,
        top_p=TOP_P,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    return [
        tokenizer.decode(x, skip_special_tokens=True).strip()
        for x in outputs
    ]

seeded_rows = []

for _, row in selected_scaffolds.iterrows():
    generated = generate_seeded(
        row["Scaffold"],
        N_PER_SEED,
        MAX_NEW_TOKENS_SEEDED
    )

    for s in generated:
        seeded_rows.append({
            "Seed_ID": row["Seed_ID"],
            "Seed_Scaffold": row["Scaffold"],
            "SMILES": s
        })

df_seeded_raw = pd.DataFrame(seeded_rows)

valid_rows = []
for _, row in df_seeded_raw.iterrows():
    mol = to_mol(row["SMILES"])
    if mol is not None:
        valid_rows.append({
            "Seed_ID": row["Seed_ID"],
            "Seed_Scaffold": row["Seed_Scaffold"],
            "SMILES": Chem.MolToSmiles(mol),
            "Mol": mol
        })

df_seeded = (
    pd.DataFrame(valid_rows)
    if valid_rows
    else pd.DataFrame(columns=["Seed_ID","Seed_Scaffold","SMILES","Mol"])
)

if not df_seeded.empty:
    df_seeded = df_seeded.drop_duplicates("SMILES").reset_index(drop=True)

print("Raw seeded outputs:", len(df_seeded_raw))
print("Valid unique seeded molecules:", len(df_seeded))
display(df_seeded.head(10))

Raw seeded outputs: 200
Valid unique seeded molecules: 34


[09:16:43] SMILES Parse Error: extra open parentheses while parsing: c1cncc(-c2c(-c3cccc4nnsc34)oc3cncc(-c4cnn(C5CCCCC5)c4)c23)c1NC[C@@H](O)[C@@H](O[C@H]2O[C@@H](CO)[C@@H](O)[C@@H](O)[C@H]2O[C@H]2O[C@H](CO)[C@@H](O
[09:16:43] SMILES Parse Error: check for mistakes around position 78:
[09:16:43] )c1NC[C@@H](O)[C@@H](O[C@H]2O[C@@H](CO)[C
[09:16:43] ~~~~~~~~~~~~~~~~~~~~^
[09:16:43] SMILES Parse Error: extra open parentheses while parsing: c1cncc(-c2c(-c3cccc4nnsc34)oc3cncc(-c4cnn(C5CCCCC5)c4)c23)c1NC[C@@H](O)[C@@H](O[C@H]2O[C@@H](CO)[C@@H](O)[C@@H](O)[C@H]2O[C@H]2O[C@H](CO)[C@@H](O
[09:16:43] SMILES Parse Error: check for mistakes around position 144:
[09:16:43] @H]2O[C@H](CO)[C@@H](O
[09:16:43] ~~~~~~~~~~~~~~~~~~~~^
[09:16:43] SMILES Parse Error: Failed parsing SMILES 'c1cncc(-c2c(-c3cccc4nnsc34)oc3cncc(-c4cnn(C5CCCCC5)c4)c23)c1NC[C@@H](O)[C@@H](O[C@H]2O[C@@H](CO)[C@@H](O)[C@@H](O)[C@H]2O[C@H]2O[C@H](CO)[C@@H](O' for input: 'c1cncc(-c2c(-c3cccc4nnsc34)oc3cncc(-c4cnn(C5CCCCC5)c4)c23)c1NC[

,Seed_ID,Seed_Scaffold,SMILES,Mol
0,Seed_01,O=C1/C=C\CCOC(=O)C2=C(/C=C\CCC1)CC=CC2=O,O=C1/C=C\CCOC(=O)C2=C(/C=C\CCC1)CC=CC2=O,<rdkit.Chem.rdchem.Mol object at 0x7c20d0298d60>
1,Seed_02,c1ncc2cc[nH]c2n1,c1ncc2cc[nH]c2n1,<rdkit.Chem.rdchem.Mol object at 0x7c20d02999a0>
2,Seed_03,c1csc(-c2ccc(CN3CCNCC3)cc2)c1,c1csc(-c2ccc(CN3CCNCC3)cc2)c1,<rdkit.Chem.rdchem.Mol object at 0x7c20d0299e00>
3,Seed_04,O=C(N=c1[nH]c2ccccc2[nH]1)c1ccccc1,O=C(N=c1[nH]c2ccccc2[nH]1)c1ccccc1,<rdkit.Chem.rdchem.Mol object at 0x7c20d029a260>
4,Seed_04,O=C(N=c1[nH]c2ccccc2[nH]1)c1ccccc1,O=C(Nc1ccccc1C(=O)N=c1[nH]c2ccccc2[nH]1)C(=O)N...,<rdkit.Chem.rdchem.Mol object at 0x7c20d029a5e0>
5,Seed_05,c1cncc(-c2c(-c3cccc4nnsc34)oc3cncc(-c4cnn(C5CC...,c1cncc(-c2c(-c3cccc4nnsc34)oc3cncc(-c4cnn(C5CC...,<rdkit.Chem.rdchem.Mol object at 0x7c20d029a730>
6,Seed_05,c1cncc(-c2c(-c3cccc4nnsc34)oc3cncc(-c4cnn(C5CC...,OC[C@@H]1O[C@H](OB(O)c2ccncc2-c2c(-c3cccc4nnsc...,<rdkit.Chem.rdchem.Mol object at 0x7c20d029a8f0>
7,Seed_06,O=C(CCCC[C@@H]1SC[C@@H]2NC(=O)N[C@@H]21)NCCOCC...,O=C(CCCC[C@@H]1SC[C@@H]2NC(=O)N[C@@H]21)NCCOCC...,<rdkit.Chem.rdchem.Mol object at 0x7c20d029ac00>
8,Seed_06,O=C(CCCC[C@@H]1SC[C@@H]2NC(=O)N[C@@H]21)NCCOCC...,O=C(CCCC[C@@H]1SC[C@@H]2NC(=O)N[C@@H]21)NCCOCC...,<rdkit.Chem.rdchem.Mol object at 0x7c20d029ad50>
9,Seed_07,O=C1Nc2ccccc2OC[C@@H]1N1CCc2cn(CC3CCS(=O)(=O)C...,O=C1Nc2ccccc2OC[C@@H]1N1CCc2cn(CC3CCS(=O)(=O)C...,<rdkit.Chem.rdchem.Mol object at 0x7c20d029b060>


## Experiment 1 — Measure scaffold retention

We **measure** whether the generated molecule still contains the BM seed. We do not assume that SMILES continuation preserves it.

In [ ]:
def add_seed_retention(df):
    df = df.copy()
    if df.empty:
        df["Seed_Retained"] = pd.Series(dtype=bool)
        return df

    retained = []
    for _, row in df.iterrows():
        seed_mol = Chem.MolFromSmiles(row["Seed_Scaffold"])
        retained.append(
            seed_mol is not None and row["Mol"].HasSubstructMatch(seed_mol)
        )

    df["Seed_Retained"] = retained
    return df

df_seeded = add_seed_retention(df_seeded)

print(
    "Experiment 1 scaffold-retention rate:",
    df_seeded["Seed_Retained"].mean() if not df_seeded.empty else np.nan
)

Experiment 1 scaffold-retention rate: 1.0


# Experiment 2 — Label-aware scaffold decoration by R-group recombination

We do not ask MolGPT to produce incomplete fragment SMILES.

Instead:

1. identify reference TAK1 molecules containing each selected BM scaffold;
2. use RDKit R-group decomposition;
3. preserve the attachment labels (`[*:1]`, `[*:2]`, ...);
4. report how many attachment positions are actually observed for each scaffold;
5. collect substituents separately for each position;
6. recombine substituents only at their matching labelled positions;
7. sanitize and deduplicate complete products.

**Novelty** comes from new scaffold–substituent combinations. Individual substituents may already occur in the reference set.


## 8. Build a robust substituent library with RDKit

For each selected scaffold, RDKit finds molecules in the reference dataset containing that scaffold and extracts their R-groups.

We use RDKit's R-group decomposition rather than trying to parse SMILES strings manually.

In [ ]:
from rdkit.Chem import rdRGroupDecomposition
from itertools import product as itertools_product

def get_matching_molecules(scaffold_smiles):
    scaffold = Chem.MolFromSmiles(scaffold_smiles)
    if scaffold is None:
        return []
    return [mol for mol in tak1_df["Mol"] if mol.HasSubstructMatch(scaffold)]

def extract_labelled_rgroups(scaffold_smiles):
    """Return the RDKit-labelled core and substituent options by attachment label."""
    scaffold = Chem.MolFromSmiles(scaffold_smiles)
    matches = get_matching_molecules(scaffold_smiles)

    if scaffold is None or not matches:
        return None, {}

    try:
        rows, unmatched = rdRGroupDecomposition.RGroupDecompose(
            [scaffold], matches, asSmiles=True
        )
    except Exception:
        return None, {}

    core_smiles = rows[0].get("Core") if rows else None
    options = {}

    for row in rows:
        for label, smi in row.items():
            if label == "Core" or not smi:
                continue

            mol = Chem.MolFromSmiles(smi)
            if mol is None or mol.GetNumHeavyAtoms() == 0 or mol.GetNumHeavyAtoms() > 12:
                continue

            dummy_atoms = [a for a in mol.GetAtoms() if a.GetAtomicNum() == 0]
            if len(dummy_atoms) != 1:
                continue

            map_num = dummy_atoms[0].GetAtomMapNum()
            if map_num <= 0:
                continue

            try:
                Chem.SanitizeMol(mol)
                canonical = Chem.MolToSmiles(mol)
            except Exception:
                continue

            options.setdefault(map_num, set()).add(canonical)

    return core_smiles, {k: sorted(v) for k, v in options.items()}

attachment_rows = []
labelled_rgroup_library = {}

for _, row in selected_scaffolds.iterrows():
    seed_id = row["Seed_ID"]
    scaffold = row["Scaffold"]
    core_labelled, options = extract_labelled_rgroups(scaffold)

    labelled_rgroup_library[seed_id] = {
        "Core_Labelled": core_labelled,
        "Options": options
    }

    attachment_rows.append({
        "Seed_ID": seed_id,
        "Seed_Scaffold": scaffold,
        "Labelled_Core": core_labelled,
        "Observed_Attachment_Points": len(options),
        "Attachment_Labels": ", ".join(f"R{k}" for k in sorted(options)) if options else "None",
        "Substituent_Options_Total": sum(len(v) for v in options.values())
    })

attachment_summary = pd.DataFrame(attachment_rows)

print("Attachment-point summary:")
display(attachment_summary)

fragment_rows = []
for seed_id, data in labelled_rgroup_library.items():
    scaffold = selected_scaffolds.loc[
        selected_scaffolds["Seed_ID"] == seed_id, "Scaffold"
    ].iloc[0]
    for map_num, choices in data["Options"].items():
        for frag in choices:
            fragment_rows.append({
                "Seed_ID": seed_id,
                "Seed_Scaffold": scaffold,
                "Attachment_Label": f"R{map_num}",
                "Attachment_MapNum": map_num,
                "Fragment_SMILES": frag
            })

fragment_df = pd.DataFrame(fragment_rows)

if fragment_df.empty:
    print("No labelled R-group fragments were extracted.")
else:
    fragment_df = fragment_df.drop_duplicates().reset_index(drop=True)
    print("Unique labelled substituent records:", len(fragment_df))
    display(fragment_df.head(20))


[09:16:43] WARNING: not removing hydrogen atom with dummy atom neighbors
[09:16:43] WARNING: not removing hydrogen atom with dummy atom neighbors
[09:16:43] WARNING: not removing hydrogen atom with dummy atom neighbors
[09:16:43] WARNING: not removing hydrogen atom with dummy atom neighbors
[09:16:43] WARNING: not removing hydrogen atom with dummy atom neighbors
[09:16:43] WARNING: not removing hydrogen atom with dummy atom neighbors
[09:16:43] WARNING: not removing hydrogen atom with dummy atom neighbors
[09:16:43] WARNING: not removing hydrogen atom with dummy atom neighbors
[09:16:43] WARNING: not removing hydrogen atom with dummy atom neighbors
[09:16:43] WARNING: not removing hydrogen atom with dummy atom neighbors
[09:16:43] WARNING: not removing hydrogen atom with dummy atom neighbors
[09:16:43] WARNING: not removing hydrogen atom with dummy atom neighbors
[09:16:43] WARNING: not removing hydrogen atom with dummy atom neighbors
[09:16:43] WARNING: not removing hydrogen atom with

Attachment-point summary:


,Seed_ID,Seed_Scaffold,Labelled_Core,Observed_Attachment_Points,Attachment_Labels,Substituent_Options_Total
0,Seed_01,O=C1/C=C\CCOC(=O)C2=C(/C=C\CCC1)CC=CC2=O,O=C1C=C([*:1])C([*:2])C2=C1C(=O)OC([*:5])C/C=C...,4,"R1, R3, R4, R5",4
1,Seed_02,c1ncc2cc[nH]c2n1,n1c([*:4])nc2[nH]c([*:1])c([*:2])c2c1[*:3],3,"R1, R2, R3",24
2,Seed_03,c1csc(-c2ccc(CN3CCNCC3)cc2)c1,c1cc(-c2cc([*:3])c([*:2])s2)ccc1CN1CCN([*:1])CC1,3,"R1, R2, R3",3
3,Seed_04,O=C(N=c1[nH]c2ccccc2[nH]1)c1ccccc1,O=C(/N=c1\[nH]c2ccccc2n1[*:2])c1cccc([*:1])c1,2,"R1, R2",2
4,Seed_05,c1cncc(-c2c(-c3cccc4nnsc34)oc3cncc(-c4cnn(C5CC...,c1cncc(-c2c(-c3cccc4nnsc34)oc3c([*:2])ncc(-c4c...,2,"R1, R2",2
5,Seed_06,O=C(CCCC[C@@H]1SC[C@@H]2NC(=O)N[C@@H]21)NCCOCC...,O=C(CCCC[C@@H]1SC[C@@H]2NC(=O)N[C@@H]21)NCCOCC...,2,"R1, R2",2
6,Seed_07,O=C1Nc2ccccc2OC[C@@H]1N1CCc2cn(CC3CCS(=O)(=O)C...,O=C1c2nn(CC3CCS(=O)(=O)CC3)cc2CCN1[C@H]1COc2cc...,1,R1,1
7,Seed_08,c1cc2[nH]ncc2cc1-c1cnc2ccc(N3C[C@@H]4C[C@H]3CO...,c1cc2[nH]nc([*:1])c2cc1-c1cnc2ccc(N3C[C@@H]4C[...,1,R1,1
8,Seed_09,O=C1NCc2c1c1c3ccccc3n3c1c1c2c2ccccc2n1C1CCC[C@...,O=C1NCc2c1c1c3ccccc3n3c1c1c2c2ccccc2n1C1([*:3]...,3,"R1, R2, R3",3
9,Seed_10,O=C(Nc1cccc(-c2nnc[nH]2)n1)Nc1nccc2ccccc12,O=C(Nc1cccc(-c2nncn2[*:1])n1)Nc1nccc2ccccc12,1,R1,2


Unique labelled substituent records: 74


,Seed_ID,Seed_Scaffold,Attachment_Label,Attachment_MapNum,Fragment_SMILES
0,Seed_01,O=C1/C=C\CCOC(=O)C2=C(/C=C\CCC1)CC=CC2=O,R1,1,CO[*:1]
1,Seed_01,O=C1/C=C\CCOC(=O)C2=C(/C=C\CCC1)CC=CC2=O,R3,3,O[*:3]
2,Seed_01,O=C1/C=C\CCOC(=O)C2=C(/C=C\CCC1)CC=CC2=O,R4,4,O[*:4]
3,Seed_01,O=C1/C=C\CCOC(=O)C2=C(/C=C\CCC1)CC=CC2=O,R5,5,C[*:5]
4,Seed_02,c1ncc2cc[nH]c2n1,R2,2,CCC(=O)/C=C/[*:2]
5,Seed_02,c1ncc2cc[nH]c2n1,R2,2,CCNC(=O)/C=C/[*:2]
6,Seed_02,c1ncc2cc[nH]c2n1,R2,2,CN(C)C(=O)/C=C/[*:2]
7,Seed_02,c1ncc2cc[nH]c2n1,R2,2,CNC(=O)/C=C/[*:2]
8,Seed_02,c1ncc2cc[nH]c2n1,R2,2,COC(=O)/C(C)=C/[*:2]
9,Seed_02,c1ncc2cc[nH]c2n1,R2,2,COC(=O)/C=C/[*:2]


## 9. A safer alternative for attachment: use RDKit's R-group recombination

Rather than manually adding bonds and risking incorrect valence handling, we use a chemically explicit `*` attachment point.

Each extracted R-group is converted into a substituent containing one dummy atom (`*`), and RDKit combines it with the core.

In [ ]:
def _dummy_info(mol):
    """Map attachment label -> (dummy atom, neighbor atom, bond type)."""
    info = {}
    for atom in mol.GetAtoms():
        if atom.GetAtomicNum() != 0:
            continue
        label = atom.GetAtomMapNum()
        neighbors = list(atom.GetNeighbors())
        if label <= 0 or len(neighbors) != 1:
            continue
        neighbor = neighbors[0]
        bond = mol.GetBondBetweenAtoms(atom.GetIdx(), neighbor.GetIdx())
        info[label] = (atom.GetIdx(), neighbor.GetIdx(), bond.GetBondType())
    return info

def recombine_labelled(core_smiles, fragment_smiles_by_label):
    """Attach each R-group to the scaffold atom carrying the same RDKit label."""
    core = Chem.MolFromSmiles(core_smiles)
    if core is None:
        return None

    core_info = _dummy_info(core)
    if not core_info or not set(fragment_smiles_by_label).issubset(core_info):
        return None

    combined = Chem.RWMol(core)
    pending = []

    for label, frag_smiles in fragment_smiles_by_label.items():
        frag = Chem.MolFromSmiles(frag_smiles)
        if frag is None:
            return None

        frag_info = _dummy_info(frag)
        if label not in frag_info:
            return None

        core_dummy, core_neighbor, core_bond = core_info[label]
        frag_dummy, frag_neighbor, _ = frag_info[label]

        offset = combined.GetNumAtoms()
        combined.InsertMol(frag)

        pending.append((
            core_neighbor,
            offset + frag_neighbor,
            core_dummy,
            offset + frag_dummy,
            core_bond
        ))

    for core_neighbor, frag_neighbor, _, _, bond_type in pending:
        combined.AddBond(core_neighbor, frag_neighbor, bond_type)

    # Remove every dummy atom after all connections have been created.
    dummy_indices = [
        atom.GetIdx() for atom in combined.GetAtoms()
        if atom.GetAtomicNum() == 0
    ]
    for idx in sorted(dummy_indices, reverse=True):
        combined.RemoveAtom(idx)

    product = combined.GetMol()
    try:
        Chem.SanitizeMol(product)
        return product
    except Exception:
        return None

def generate_labelled_combinations(seed_id, core_smiles, options, max_combinations=50):
    labels = sorted(options)
    if not labels:
        return []

    choices = [options[label] for label in labels]
    theoretical_total = int(np.prod([len(x) for x in choices]))
    combinations = list(itertools_product(*choices))

    if len(combinations) > max_combinations:
        rng = random.Random(SEED + int(seed_id.split("_")[-1]))
        combinations = rng.sample(combinations, max_combinations)

    products = []
    for combo in combinations:
        by_label = dict(zip(labels, combo))
        mol = recombine_labelled(core_smiles, by_label)
        if mol is not None:
            products.append((by_label, mol, theoretical_total))
    return products


## 10. Generate Experiment 2 molecules

Because the extracted R-groups come from real, valid molecules, this step avoids the incomplete-SMILES problem that occurred when we asked MolGPT to produce arbitrary fragments.

The resulting complete molecules are still genuinely generated **combinations**: the scaffold is fixed, while substituents are recombined.

In [ ]:
MAX_COMBINATIONS_PER_SCAFFOLD = 50

decorated_rows = []

for _, seed_row in selected_scaffolds.iterrows():
    seed_id = seed_row["Seed_ID"]
    scaffold = seed_row["Scaffold"]

    data = labelled_rgroup_library.get(seed_id, {})
    core_labelled = data.get("Core_Labelled")
    options = data.get("Options", {})

    if not core_labelled or not options:
        continue

    generated = generate_labelled_combinations(
        seed_id, core_labelled, options,
        max_combinations=MAX_COMBINATIONS_PER_SCAFFOLD
    )

    for by_label, product_mol, theoretical_total in generated:
        decorated_rows.append({
            "Seed_ID": seed_id,
            "Seed_Scaffold": scaffold,
            "Fragment_SMILES": " | ".join(
                f"R{label}={frag}" for label, frag in sorted(by_label.items())
            ),
            "SMILES": Chem.MolToSmiles(product_mol),
            "Mol": product_mol,
            "Attachment_Combination": ";".join(f"R{label}" for label in sorted(by_label)),
            "Theoretical_Combinations": theoretical_total
        })

df_decorated = pd.DataFrame(decorated_rows)

if df_decorated.empty:
    df_decorated = pd.DataFrame(columns=[
        "Seed_ID","Seed_Scaffold","Fragment_SMILES","SMILES","Mol",
        "Attachment_Combination","Theoretical_Combinations"
    ])
else:
    df_decorated = df_decorated.drop_duplicates("SMILES").reset_index(drop=True)

print("Experiment 2 valid unique molecules:", len(df_decorated))
display(df_decorated.head(20))


Experiment 2 valid unique molecules: 68


,Seed_ID,Seed_Scaffold,Fragment_SMILES,SMILES,Mol,Attachment_Combination,Theoretical_Combinations
0,Seed_01,O=C1/C=C\CCOC(=O)C2=C(/C=C\CCC1)CC=CC2=O,R1=CO[*:1] | R3=O[*:3] | R4=O[*:4] | R5=C[*:5],COC1=CC(=O)C2=C(/C=C\CC(O)C(O)C(=O)/C=C\CC(C)O...,<rdkit.Chem.rdchem.Mol object at 0x7c20d00e85f0>,R1;R3;R4;R5,1
1,Seed_02,c1ncc2cc[nH]c2n1,R1=C[*:1] | R2=NC(=O)C#C[*:2] | R3=FC1(F)CCC(O...,Cc1[nH]c2ncnc(OC3CCC(F)(F)CC3)c2c1C#CC(N)=O,<rdkit.Chem.rdchem.Mol object at 0x7c20d00e8d60>,R1;R2;R3,132
2,Seed_02,c1ncc2cc[nH]c2n1,R1=C[*:1] | R2=CN(C)C(=O)/C=C/[*:2] | R3=CCC(C...,CCC(C)Oc1ncnc2[nH]c(C)c(C=CC(=O)N(C)C)c12,<rdkit.Chem.rdchem.Mol object at 0x7c20d00e8cf0>,R1;R2;R3,132
3,Seed_02,c1ncc2cc[nH]c2n1,R1=C[*:1] | R2=CNC(=O)/C=C/[*:2] | R3=Fc1cccc(...,CNC(=O)C=Cc1c(C)[nH]c2ncnc(OCc3cccc(F)c3)c12,<rdkit.Chem.rdchem.Mol object at 0x7c20d00e8e40>,R1;R2;R3,132
4,Seed_02,c1ncc2cc[nH]c2n1,R1=C[*:1] | R2=NC(=O)C#C[*:2] | R3=CC(C)(C)CO[...,Cc1[nH]c2ncnc(OCC(C)(C)C)c2c1C#CC(N)=O,<rdkit.Chem.rdchem.Mol object at 0x7c20d00e82e0>,R1;R2;R3,132
5,Seed_02,c1ncc2cc[nH]c2n1,R1=C[*:1] | R2=COC(=O)/C(C)=C/[*:2] | R3=Fc1cc...,COC(=O)C(C)=Cc1c(C)[nH]c2ncnc(OCc3cccc(F)c3)c12,<rdkit.Chem.rdchem.Mol object at 0x7c20d00e8f90>,R1;R2;R3,132
6,Seed_02,c1ncc2cc[nH]c2n1,R1=C[*:1] | R2=CNC(=O)/C=C/[*:2] | R3=CC(C)(C)...,CNC(=O)C=Cc1c(C)[nH]c2ncnc(OCC(C)(C)C)c12,<rdkit.Chem.rdchem.Mol object at 0x7c20d00e9000>,R1;R2;R3,132
7,Seed_02,c1ncc2cc[nH]c2n1,R1=C[*:1] | R2=CCC(=O)/C=C/[*:2] | R3=CC(C)CO[...,CCC(=O)C=Cc1c(C)[nH]c2ncnc(OCC(C)C)c12,<rdkit.Chem.rdchem.Mol object at 0x7c20d00e8f20>,R1;R2;R3,132
8,Seed_02,c1ncc2cc[nH]c2n1,R1=C[*:1] | R2=CN(C)C(=O)/C=C/[*:2] | R3=CC(C)...,Cc1[nH]c2ncnc(OC(C)C)c2c1C=CC(=O)N(C)C,<rdkit.Chem.rdchem.Mol object at 0x7c20d00e8dd0>,R1;R2;R3,132
9,Seed_02,c1ncc2cc[nH]c2n1,R1=C[*:1] | R2=CCNC(=O)/C=C/[*:2] | R3=CC(C)(C...,CCNC(=O)C=Cc1c(C)[nH]c2ncnc(OC(C)(C)C)c12,<rdkit.Chem.rdchem.Mol object at 0x7c20d00e90e0>,R1;R2;R3,132


## 11. Verify scaffold retention in Experiment 2

In [ ]:
if not df_decorated.empty:
    df_decorated["Seed_Retained"] = [
        mol.HasSubstructMatch(Chem.MolFromSmiles(core))
        for mol, core in zip(df_decorated["Mol"], df_decorated["Seed_Scaffold"])
    ]

    print("Experiment 2 scaffold-retention rate:", df_decorated["Seed_Retained"].mean())

    print("\nAttachment combinations represented:")
    display(
        df_decorated["Attachment_Combination"]
        .value_counts()
        .rename_axis("Attachment_Combination")
        .reset_index(name="Molecule_Count")
        .head(20)
    )
else:
    print("No Experiment 2 molecules were produced.")


Experiment 2 scaffold-retention rate: 1.0

Attachment combinations represented:


,Attachment_Combination,Molecule_Count
0,R1;R2;R3,52
1,R1;R2,7
2,R1,6
3,R1;R3;R4;R5,1
4,R2;R3;R4;R5;R6,1
5,R1;R2;R3;R4;R5,1


## What determines the Experiment 2 generation space?

The number of attachment positions is **measured from the reference TAK1 chemistry**; it is not guessed by MolGPT.

For a scaffold with one observed position:

`R1 ∈ {A, B, C}`

there are up to 3 decoration choices.

For a scaffold with two observed positions:

`R1 ∈ {A, B, C}`  
`R2 ∈ {D, E}`

there are up to `3 × 2 = 6` combinations.

The notebook reports this explicitly in `attachment_summary` and limits the number of generated combinations per scaffold with `MAX_COMBINATIONS_PER_SCAFFOLD`.

Thus:
- the **scaffold** defines the topology;
- RDKit defines the **attachment labels**;
- the reference molecules define the **available substituent options**;
- recombination creates the new complete molecules;
- RDKit validates the products.


# Shared analysis

We now apply the same basic checks to all three populations.

## 12. Molecular descriptors

In [ ]:
def add_descriptors(df):
    df = df.copy()
    if df.empty:
        return df

    df["MW"] = df["Mol"].apply(Descriptors.MolWt)
    df["LogP"] = df["Mol"].apply(Crippen.MolLogP)
    df["TPSA"] = df["Mol"].apply(rdMolDescriptors.CalcTPSA)
    df["HBD"] = df["Mol"].apply(Lipinski.NumHDonors)
    df["HBA"] = df["Mol"].apply(Lipinski.NumHAcceptors)
    df["RotBonds"] = df["Mol"].apply(Lipinski.NumRotatableBonds)
    df["QED"] = df["Mol"].apply(QED.qed)
    return df

df_uncond = add_descriptors(df_uncond)
df_seeded = add_descriptors(df_seeded)
df_decorated = add_descriptors(df_decorated)

display(
    pd.DataFrame({
        "Unconditional": df_uncond[["MW","LogP","TPSA","QED"]].mean()
        if not df_uncond.empty else np.nan,
        "Experiment 1": df_seeded[["MW","LogP","TPSA","QED"]].mean()
        if not df_seeded.empty else np.nan,
        "Experiment 2": df_decorated[["MW","LogP","TPSA","QED"]].mean()
        if not df_decorated.empty else np.nan
    }).round(3)
)

,Unconditional,Experiment 1,Experiment 2
MW,335.074,454.462,360.830
LogP,-0.500,2.799,2.942
TPSA,105.355,123.182,92.408
QED,0.655,0.386,0.670


## 13. Exact novelty relative to the known TAK1 dataset

In [ ]:
known = set(tak1_df["SMILES"])

for df in [df_uncond, df_seeded, df_decorated]:
    if not df.empty:
        df["Novel_vs_Known_TAK1"] = ~df["SMILES"].isin(known)

print("Exactly novel molecules:")
print("Unconditional:", int(df_uncond["Novel_vs_Known_TAK1"].sum()) if not df_uncond.empty else 0)
print("Experiment 1:", int(df_seeded["Novel_vs_Known_TAK1"].sum()) if not df_seeded.empty else 0)
print("Experiment 2:", int(df_decorated["Novel_vs_Known_TAK1"].sum()) if not df_decorated.empty else 0)

Exactly novel molecules:
Unconditional: 100
Experiment 1: 34
Experiment 2: 57


## 14. Lipinski + PAINS filtering

In [ ]:
params = FilterCatalogParams()
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS_A)
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS_B)
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS_C)
pains = FilterCatalog(params)

def quality_filter(df):
    if df.empty:
        return df.copy()

    df = df.copy()
    df["LipinskiPass"] = (
        (df["MW"] <= 500) &
        (df["LogP"] <= 5) &
        (df["HBD"] <= 5) &
        (df["HBA"] <= 10)
    )
    df["PAINS"] = df["Mol"].apply(pains.HasMatch)
    df["QualityPass"] = df["LipinskiPass"] & ~df["PAINS"]
    return df

df_uncond = quality_filter(df_uncond)
df_seeded = quality_filter(df_seeded)
df_decorated = quality_filter(df_decorated)

final_uncond = df_uncond[df_uncond["QualityPass"]].copy()
final_seeded = df_seeded[df_seeded["QualityPass"]].copy()
final_decorated = df_decorated[df_decorated["QualityPass"]].copy()

print("Final unconditional:", len(final_uncond))
print("Final Experiment 1:", len(final_seeded))
print("Final Experiment 2:", len(final_decorated))

Final unconditional: 100
Final Experiment 1: 16
Final Experiment 2: 63


## 15. Chemical similarity to known TAK1 inhibitors

In [ ]:
tak1_fps = [fp_gen.GetFingerprint(m) for m in tak1_df["Mol"]]

def max_similarity(mol):
    if mol is None or not tak1_fps:
        return np.nan
    fp = fp_gen.GetFingerprint(mol)
    return max(DataStructs.TanimotoSimilarity(fp, x) for x in tak1_fps)

for df in [df_uncond, df_seeded, df_decorated]:
    if not df.empty:
        df["Max_TAK1_Tanimoto"] = df["Mol"].apply(max_similarity)

summary = pd.DataFrame({
    "Unconditional": {
        "Raw": len(raw_uncond),
        "Valid unique": len(df_uncond),
        "Quality pass": len(final_uncond),
        "Mean similarity": df_uncond["Max_TAK1_Tanimoto"].mean() if not df_uncond.empty else np.nan
    },
    "Experiment 1": {
        "Raw": len(df_seeded_raw),
        "Valid unique": len(df_seeded),
        "Quality pass": len(final_seeded),
        "Mean similarity": df_seeded["Max_TAK1_Tanimoto"].mean() if not df_seeded.empty else np.nan
    },
    "Experiment 2": {
        "Raw": len(df_decorated),
        "Valid unique": len(df_decorated),
        "Quality pass": len(final_decorated),
        "Mean similarity": df_decorated["Max_TAK1_Tanimoto"].mean() if not df_decorated.empty else np.nan
    }
}).T

display(summary.round(3))

,Raw,Valid unique,Quality pass,Mean similarity
Unconditional,100.0,100.0,100.0,0.194
Experiment 1,200.0,34.0,16.0,0.449
Experiment 2,68.0,68.0,63.0,0.629


In [ ]:
# Re-apply quality filter to ensure 'Max_TAK1_Tanimoto' is propagated
# The pains object and quality_filter function are defined in cell 3BjRb3ooOXZh

df_uncond = quality_filter(df_uncond)
df_seeded = quality_filter(df_seeded)
df_decorated = quality_filter(df_decorated)

final_uncond = df_uncond[df_uncond["QualityPass"]].copy()
final_seeded = df_seeded[df_seeded["QualityPass"]].copy()
final_decorated = df_decorated[df_decorated["QualityPass"]].copy()

print("Final unconditional (re-filtered):", len(final_uncond))
print("Final Experiment 1 (re-filtered):", len(final_seeded))
print("Final Experiment 2 (re-filtered):", len(final_decorated))

Final unconditional (re-filtered): 100
Final Experiment 1 (re-filtered): 16
Final Experiment 2 (re-filtered): 63


## 16. Inspect generated molecules

These are **generation-stage candidates only**. No TAK1 docking or binding claim is made here.

In [ ]:
cols = ["SMILES","MW","LogP","TPSA","HBD","HBA","QED","Max_TAK1_Tanimoto"]

print("Baseline A")
display(final_uncond[cols].head(10) if not final_uncond.empty else pd.DataFrame())

print("Experiment 1")
display(
    final_seeded[["Seed_ID","Seed_Scaffold"] + cols].head(10)
    if not final_seeded.empty else pd.DataFrame()
)

print("Experiment 2")
display(
    final_decorated[
        ["Seed_ID","Seed_Scaffold","Fragment_SMILES"] + cols
    ].head(10)
    if not final_decorated.empty else pd.DataFrame()
)

Baseline A


,SMILES,MW,LogP,TPSA,HBD,HBA,QED,Max_TAK1_Tanimoto
0,C[C@H]1C[C@@H](NC(=O)c2cnn(C)n2)CN1C(=O)c1ccc(...,370.377,-0.64370,150.37,3,7,0.552274,0.178218
1,CCC[C@@H](O)[C@@H](CO)NC(=O)C(=O)Nc1cccc(C#N)n1,306.322,-0.47012,135.34,4,6,0.524644,0.243902
2,C[C@H](C(=O)N(C)C1CN(C(=O)Cc2ccnn2C)C1)N1C(=O)...,361.402,-0.83070,95.82,0,5,0.638034,0.181818
3,Cc1ncc(C(=O)N[C@H]2C[C@@H](NC(=O)[C@H]3COCCO3)...,322.365,-0.47908,94.48,2,5,0.780781,0.183908
4,O=C1NCC[C@H]1N1CC[C@@H](N2CCN(C(=O)CCc3cnn[nH]...,361.450,-1.15570,97.46,2,6,0.686996,0.175258
5,CNC(=O)NC(=O)CNC[C@@H]1CCCCN1C(=O)c1ccncn1,334.380,-0.48350,116.32,3,6,0.669399,0.193878
6,CC[C@@H](CNS(C)(=O)=O)NC(=O)N[C@H]1COC[C@@H]1OC,309.388,-0.97280,105.76,3,5,0.560817,0.135802
7,C[C@H](CNC(=O)c1ccc(-c2nn[nH]n2)[nH]1)N(C)CC(=...,389.420,-1.79670,152.00,4,7,0.436005,0.201923
8,CN(CC(=O)N[C@H]1CCS(=O)(=O)C1)C(=O)CNC(=O)c1cc...,419.862,-0.02940,112.65,2,5,0.672031,0.181818
9,C[C@@H]1CC[C@@H]1N[C@@H]1CCN(C(N)=O)C1,197.282,0.52750,58.36,2,2,0.678043,0.173333


Experiment 1


,Seed_ID,Seed_Scaffold,SMILES,MW,LogP,TPSA,HBD,HBA,QED,Max_TAK1_Tanimoto
0,Seed_01,O=C1/C=C\CCOC(=O)C2=C(/C=C\CCC1)CC=CC2=O,O=C1/C=C\CCOC(=O)C2=C(/C=C\CCC1)CC=CC2=O,286.327,2.6107,60.44,0,4,0.507089,0.166667
1,Seed_02,c1ncc2cc[nH]c2n1,c1ncc2cc[nH]c2n1,119.127,0.9579,41.57,1,2,0.560736,0.185185
2,Seed_03,c1csc(-c2ccc(CN3CCNCC3)cc2)c1,c1csc(-c2ccc(CN3CCNCC3)cc2)c1,258.390,2.8203,15.27,1,3,0.910475,0.312500
3,Seed_04,O=C(N=c1[nH]c2ccccc2[nH]1)c1ccccc1,O=C(N=c1[nH]c2ccccc2[nH]1)c1ccccc1,237.262,2.2371,61.01,2,1,0.669526,0.380000
4,Seed_04,O=C(N=c1[nH]c2ccccc2[nH]1)c1ccccc1,O=C(Nc1ccccc1C(=O)N=c1[nH]c2ccccc2[nH]1)C(=O)N...,423.429,-0.0128,150.88,5,5,0.375419,0.217949
9,Seed_07,O=C1Nc2ccccc2OC[C@@H]1N1CCc2cn(CC3CCS(=O)(=O)C...,O=C1Nc2ccccc2OC[C@@H]1N1CCc2cn(CC3CCS(=O)(=O)C...,444.513,1.1059,110.60,1,6,0.759293,0.704225
10,Seed_08,c1cc2[nH]ncc2cc1-c1cnc2ccc(N3C[C@@H]4C[C@H]3CO...,c1cc2[nH]ncc2cc1-c1cnc2ccc(N3C[C@@H]4C[C@H]3CO...,332.367,2.2501,71.34,1,5,0.609504,0.703125
12,Seed_10,O=C(Nc1cccc(-c2nnc[nH]2)n1)Nc1nccc2ccccc12,O=C(Nc1cccc(-c2nnc[nH]2)n1)Nc1nccc2ccccc12,331.339,3.0589,108.48,3,5,0.534454,0.606557
13,Seed_11,O=C(Nc1ccc(C(=O)N2CCNCC2)cc1C1CCCCC1)c1csc2c(=...,O=C(Nc1ccc(C(=O)N2CCNCC2)cc1C1CCCCC1)c1csc2c(=...,465.579,3.3300,107.19,3,6,0.547923,0.840580
18,Seed_14,O=c1[nH]nc2ccc3ccc(-c4ccc[nH]4)cc3n12,O=c1[nH]nc2ccc3ccc(-c4ccc[nH]4)cc3n12,250.261,2.1709,65.95,2,2,0.543415,0.627451


Experiment 2


,Seed_ID,Seed_Scaffold,Fragment_SMILES,SMILES,MW,LogP,TPSA,HBD,HBA,QED,Max_TAK1_Tanimoto
0,Seed_01,O=C1/C=C\CCOC(=O)C2=C(/C=C\CCC1)CC=CC2=O,R1=CO[*:1] | R3=O[*:3] | R4=O[*:4] | R5=C[*:5],COC1=CC(=O)C2=C(/C=C\CC(O)C(O)C(=O)/C=C\CC(C)O...,362.378,0.91490,110.13,2,7,0.526953,0.631579
1,Seed_02,c1ncc2cc[nH]c2n1,R1=C[*:1] | R2=NC(=O)C#C[*:2] | R3=FC1(F)CCC(O...,Cc1[nH]c2ncnc(OC3CCC(F)(F)CC3)c2c1C#CC(N)=O,334.326,2.05982,93.89,2,4,0.822125,0.515625
2,Seed_02,c1ncc2cc[nH]c2n1,R1=C[*:1] | R2=CN(C)C(=O)/C=C/[*:2] | R3=CCC(C...,CCC(C)Oc1ncnc2[nH]c(C)c(C=CC(=O)N(C)C)c12,302.378,2.54502,71.11,1,4,0.861622,0.516667
3,Seed_02,c1ncc2cc[nH]c2n1,R1=C[*:1] | R2=CNC(=O)/C=C/[*:2] | R3=Fc1cccc(...,CNC(=O)C=Cc1c(C)[nH]c2ncnc(OCc3cccc(F)c3)c12,340.358,2.74362,79.90,2,4,0.700059,0.552239
4,Seed_02,c1ncc2cc[nH]c2n1,R1=C[*:1] | R2=NC(=O)C#C[*:2] | R3=CC(C)(C)CO[...,Cc1[nH]c2ncnc(OCC(C)(C)C)c2c1C#CC(N)=O,286.335,1.52802,93.89,2,4,0.818508,0.465517
5,Seed_02,c1ncc2cc[nH]c2n1,R1=C[*:1] | R2=COC(=O)/C(C)=C/[*:2] | R3=Fc1cc...,COC(=O)C(C)=Cc1c(C)[nH]c2ncnc(OCc3cccc(F)c3)c12,355.369,3.56072,77.10,1,5,0.559270,0.507246
6,Seed_02,c1ncc2cc[nH]c2n1,R1=C[*:1] | R2=CNC(=O)/C=C/[*:2] | R3=CC(C)(C)...,CNC(=O)C=Cc1c(C)[nH]c2ncnc(OCC(C)(C)C)c12,302.378,2.45042,79.90,2,4,0.850191,0.500000
7,Seed_02,c1ncc2cc[nH]c2n1,R1=C[*:1] | R2=CCC(=O)/C=C/[*:2] | R3=CC(C)CO[...,CCC(=O)C=Cc1c(C)[nH]c2ncnc(OCC(C)C)c12,287.363,3.29342,67.87,1,4,0.827716,0.641509
8,Seed_02,c1ncc2cc[nH]c2n1,R1=C[*:1] | R2=CN(C)C(=O)/C=C/[*:2] | R3=CC(C)...,Cc1[nH]c2ncnc(OC(C)C)c2c1C=CC(=O)N(C)C,288.351,2.15492,71.11,1,4,0.875103,0.483333
9,Seed_02,c1ncc2cc[nH]c2n1,R1=C[*:1] | R2=CCNC(=O)/C=C/[*:2] | R3=CC(C)(C...,CCNC(=O)C=Cc1c(C)[nH]c2ncnc(OC(C)(C)C)c12,302.378,2.59292,79.90,2,4,0.850753,0.460317


## 17. Save outputs

The CSV files preserve the generation mode and seed information so the next notebook can load the candidates for TAK1 structure-guided evaluation.

In [ ]:
final_uncond.to_csv("/content/drive/MyDrive/De-Novo/MolGPT_baseline_unconditional.csv", index=False)
final_seeded.to_csv("/content/drive/MyDrive/De-Novo/MolGPT_experiment1_seeded.csv", index=False)
final_decorated.to_csv("/content/drive/MyDrive/De-Novo/MolGPT_experiment2_decorated.csv", index=False)
selected_scaffolds.to_csv("/content/drive/MyDrive/De-Novo/TAK1_diverse_20_BM_scaffolds.csv", index=False)
attachment_summary.to_csv("/content/drive/MyDrive/De-Novo/TAK1_scaffold_attachment_summary.csv", index=False)
fragment_df.to_csv("/content/drive/MyDrive/De-Novo/TAK1_labelled_rgroup_library.csv", index=False)

print("Saved six CSV files to /content/")


Saved six CSV files to /content/


# Interpretation

### Baseline A
MolGPT generates molecules without a TAK1-derived seed.

### Experiment 1
MolGPT receives a TAK1-derived BM scaffold as a SMILES prefix and performs ordinary autoregressive continuation. Scaffold retention is measured afterward.

### Experiment 2
The same BM scaffold is treated as a fixed core. RDKit determines the observed attachment positions, labels them, extracts position-specific substituents, and recombines those substituents across the same scaffold. The model is not responsible for discovering attachment points.

### Next notebook
The three generated populations can now be compared in chemical space before selecting a manageable, diverse set for TAK1 structure-guided docking.
